# 700-Wide Kata Pipeline
**Reference case:** Nordstar Customer 360 (synthetic retail dataset)  
**Stack:** DuckDB 1.5.4 + Python 3.14 + pandas + plotly  
**Date:** 2026-07-02

Cells implement K 7.W.1 through K 7.W.7. Run top-to-bottom; each cell depends on the previous.

In [ ]:
# K 7.W.1 — Set up workspace
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'duckdb>=1.4', 'pandas', 'numpy', 'matplotlib',
                'plotly', 'pyarrow', 'tabulate', 'nbformat', '--quiet'], check=True)

import duckdb, pandas as pd, numpy as np, os, random
from datetime import datetime

BASE = os.path.abspath('..')   # kata-workspace/../  =  700-wide/
for d in ['bronze', 'silver', 'gold', 'bronze2', 'silver2', 'gold2']:
    os.makedirs(os.path.join(BASE, d), exist_ok=True)

con = duckdb.connect()
con.execute("""
    CREATE TABLE hello_world (
        id INTEGER, message VARCHAR, created_at TIMESTAMP
    )
""")
con.execute(f"""
    INSERT INTO hello_world VALUES
        (1, 'Pipeline kata workspace ready', NOW()),
        (2, 'DuckDB {duckdb.__version__}', NOW()),
        (3, 'Nordstar Customer 360', NOW())
""")
display(con.execute('SELECT * FROM hello_world').fetchdf())
print('Environment ready [OK]')


In [ ]:
# K 7.W.2 — Generate bronze dataset (500 rows, deliberate quality issues)
import os, random, numpy as np, pandas as pd
BASE = os.path.abspath('..')

random.seed(42)
np.random.seed(42)

N = 500
order_ids = [f'ORD-{i:05d}' for i in range(1, N + 1)]
for i in random.sample(range(N), 15):          # ~3% duplicates
    order_ids[i] = random.choice(order_ids[:100])

customer_ids = np.random.randint(1000, 10000, N)
regions      = np.random.choice(['North','South','East','West'], N, p=[0.27,0.25,0.25,0.23])
categories   = np.random.choice(['Electronics','Clothing','Food','Home','Sports'], N)

def rand_date():
    d = pd.Timestamp('2024-01-01') + pd.Timedelta(days=random.randint(0, 364))
    return random.choice([
        d.strftime('%Y-%m-%d'), d.strftime('%d/%m/%Y'), d.strftime('%b %d %Y')])

amounts  = np.random.uniform(5.0, 500.0, N).round(2)
for i in random.sample(range(N), 25):          # 5% null
    amounts[i] = float('nan')
for i in random.sample([j for j in range(N) if not np.isnan(amounts[j])], 10):  # ~2% negative
    amounts[i] = -amounts[i]

df = pd.DataFrame({
    'order_id': order_ids, 'customer_id': customer_ids, 'region': regions,
    'order_date': [rand_date() for _ in range(N)],
    'product_category': categories, 'amount': amounts,
    'quantity': np.random.randint(1, 11, N),
    'status': np.random.choice(['completed','returned','pending'], N, p=[0.80,0.15,0.05]),
})
bronze_path = os.path.join(BASE, 'bronze', 'transactions_raw.csv')
df.to_csv(bronze_path, index=False)

import duckdb
con2 = duckdb.connect()
con2.execute(f"CREATE TABLE bronze AS SELECT * FROM read_csv_auto('{bronze_path}')")
prof = con2.execute("""
    SELECT COUNT(*) total_rows,
           SUM(CASE WHEN amount IS NULL THEN 1 ELSE 0 END) null_amount,
           (SELECT COUNT(*) FROM (
               SELECT order_id FROM bronze GROUP BY order_id HAVING COUNT(*) > 1)
           ) dup_order_ids
    FROM bronze""").fetchdf()
display(prof)
print('Date formats: 3  (YYYY-MM-DD, DD/MM/YYYY, Mon DD YYYY)')
print('bronze/transactions_raw.csv written')


In [ ]:
# K 7.W.3 — Clean bronze -> silver
import duckdb, os
BASE = os.path.abspath('..')
con3 = duckdb.connect()
bronze_path = os.path.join(BASE, 'bronze', 'transactions_raw.csv')
silver_path = os.path.join(BASE, 'silver', 'transactions_clean.parquet')
con3.execute(f"CREATE TABLE bronze AS SELECT * FROM read_csv_auto('{bronze_path}')")
bronze_count = con3.execute('SELECT COUNT(*) FROM bronze').fetchone()[0]

con3.execute("""
    CREATE TABLE silver AS
    WITH dated AS (
        SELECT *,
            COALESCE(
                TRY_STRPTIME(order_date, '%Y-%m-%d'),
                TRY_STRPTIME(order_date, '%d/%m/%Y'),
                TRY_STRPTIME(order_date, '%b %d %Y')
            )::DATE AS order_date_clean
        FROM bronze WHERE amount IS NOT NULL
    ),
    deduped AS (
        SELECT *, ROW_NUMBER() OVER (
            PARTITION BY order_id ORDER BY customer_id DESC
        ) AS rn FROM dated
    )
    SELECT order_id, customer_id, region,
           order_date_clean AS order_date, product_category,
           CAST(amount AS DOUBLE) AS amount, quantity, status
    FROM deduped WHERE rn = 1
""")
con3.execute(f"COPY silver TO '{silver_path}' (FORMAT PARQUET)")

silver_count = con3.execute('SELECT COUNT(*) FROM silver').fetchone()[0]
null_count   = con3.execute('SELECT COUNT(*) FROM bronze WHERE amount IS NULL').fetchone()[0]
# Dedup runs after null filter, so count duplicates only among non-null rows
dup_removed  = con3.execute("""
    SELECT COALESCE(SUM(cnt-1),0) FROM (
        SELECT COUNT(*) cnt FROM bronze
        WHERE amount IS NOT NULL
        GROUP BY order_id HAVING COUNT(*) > 1)
""").fetchone()[0]
expected     = bronze_count - null_count - int(dup_removed)

verify = con3.execute("""
    SELECT COUNT(*) total,
           SUM(CASE WHEN amount IS NULL THEN 1 ELSE 0 END) null_amount,
           (SELECT COUNT(*) FROM (
               SELECT order_id FROM silver GROUP BY order_id HAVING COUNT(*) > 1)) dup_ids,
           SUM(CASE WHEN order_date IS NULL THEN 1 ELSE 0 END) null_dates,
           SUM(CASE WHEN amount < 0 THEN 1 ELSE 0 END) neg_amounts
    FROM silver""").fetchdf()
display(verify)
print(f'Math: {bronze_count} - {null_count} nulls - {int(dup_removed)} dups = {expected} | actual {silver_count} => {"PASS" if silver_count == expected else "FAIL"}')


In [ ]:
# K 7.W.4 — Build gold metrics
import duckdb, os, pandas as pd
BASE = os.path.abspath('..')
con4 = duckdb.connect()
con4.execute(f"CREATE TABLE silver AS SELECT * FROM read_parquet('{os.path.join(BASE, 'silver', 'transactions_clean.parquet')}')")

# Table 1: daily_sales_by_category — grain: date x region x category, completed only
con4.execute("""
    CREATE TABLE daily_sales AS
    SELECT order_date, region, product_category,
           ROUND(SUM(amount), 2) AS total_revenue,
           COUNT(DISTINCT order_id) AS order_count
    FROM silver WHERE status = 'completed'
    GROUP BY order_date, region, product_category
    HAVING SUM(amount) > 0
    ORDER BY order_date, region, product_category
""")
con4.execute(f"COPY daily_sales TO '{os.path.join(BASE, 'gold', 'daily_sales_by_category.parquet')}' (FORMAT PARQUET)")

# Table 2: returns_rate — grain: date, denominator = completed + returned (not pending)
con4.execute("""
    CREATE TABLE returns_rate AS
    SELECT order_date,
           COUNT(DISTINCT CASE WHEN status IN ('completed','returned') THEN order_id END) AS total_orders,
           COUNT(DISTINCT CASE WHEN status = 'returned' THEN order_id END) AS returned_orders,
           COALESCE(ROUND(
               COUNT(DISTINCT CASE WHEN status = 'returned' THEN order_id END) * 100.0 /
               NULLIF(COUNT(DISTINCT CASE WHEN status IN ('completed','returned') THEN order_id END), 0),
           2), 0.0) AS returns_rate_pct
    FROM silver GROUP BY order_date ORDER BY order_date
""")
con4.execute(f"COPY returns_rate TO '{os.path.join(BASE, 'gold', 'returns_rate.parquet')}' (FORMAT PARQUET)")

# Verification
total = con4.execute('SELECT COUNT(*) FROM daily_sales').fetchone()[0]
unique = con4.execute("SELECT COUNT(DISTINCT order_date||'|'||region||'|'||product_category) FROM daily_sales").fetchone()[0]
spot_date = con4.execute('SELECT order_date FROM returns_rate ORDER BY returned_orders DESC LIMIT 1').fetchone()[0]
ret_m = con4.execute(f"SELECT COUNT(DISTINCT order_id) FROM silver WHERE order_date='{spot_date}' AND status='returned'").fetchone()[0]
tot_m = con4.execute(f"SELECT COUNT(DISTINCT order_id) FROM silver WHERE order_date='{spot_date}' AND status IN ('completed','returned')").fetchone()[0]
rate_m = round(ret_m / tot_m * 100, 2) if tot_m else 0.0
rate_g = con4.execute(f"SELECT returns_rate_pct FROM returns_rate WHERE order_date='{spot_date}'").fetchone()[0]
print(f'Grain check: {total} rows == {unique} unique combos => {"PASS" if total==unique else "FAIL"}')
print(f'Spot-check {spot_date}: manual={rate_m}% gold={rate_g}% => {"PASS" if rate_m==rate_g else "FAIL"}')
zero_null = con4.execute('SELECT COUNT(*) FROM returns_rate WHERE returned_orders=0 AND returns_rate_pct IS NULL').fetchone()[0]
print(f'Zero-return NULL check: {"PASS" if zero_null==0 else f"FAIL ({zero_null})"}')


In [ ]:
# K 7.W.5 — DQ checks with break-and-verify
import duckdb, os
BASE = os.path.abspath('..')
con5 = duckdb.connect()
con5.execute(f"CREATE TABLE gold_sales AS SELECT * FROM read_parquet('{os.path.join(BASE, 'gold', 'daily_sales_by_category.parquet')}')")
con5.execute(f"CREATE TABLE gold_returns AS SELECT * FROM read_parquet('{os.path.join(BASE, 'gold', 'returns_rate.parquet')}')")

def run_all_checks(label=''):
    results = []
    def chk(name, sql, expect_zero=True):
        v = con5.execute(sql).fetchone()[0]
        ok = (v == 0) if expect_zero else (v > 0)
        results.append((name, ok))
        print(f'  {"OK" if ok else "XX"}  {name}: {"PASS" if ok else f"FAIL ({v})"}')
    print(f'\n=== DQ {label} ===')
    chk('1 No null date/region/category', 'SELECT COUNT(*) FROM gold_sales WHERE order_date IS NULL OR region IS NULL OR product_category IS NULL')
    chk('2 total_revenue > 0',            'SELECT COUNT(*) FROM gold_sales WHERE total_revenue <= 0')
    chk('3 order_count > 0',              'SELECT COUNT(*) FROM gold_sales WHERE order_count <= 0')
    chk('4 No duplicate grain',           'SELECT COUNT(*) FROM (SELECT order_date,region,product_category FROM gold_sales GROUP BY 1,2,3 HAVING COUNT(*)>1)')
    chk('5 No null date in returns',      'SELECT COUNT(*) FROM gold_returns WHERE order_date IS NULL')
    mn,mx = con5.execute('SELECT MIN(returns_rate_pct),MAX(returns_rate_pct) FROM gold_returns').fetchone()
    ok6 = 0.0 <= mn and mx <= 100.0
    results.append(('6 rate in [0,100]', ok6))
    print(f'  {"OK" if ok6 else "XX"}  6 rate in [0,100]: {"PASS" if ok6 else "FAIL"} ({mn}..{mx})')
    chk('7 returned<=total',              'SELECT COUNT(*) FROM gold_returns WHERE returned_orders > total_orders')
    span = con5.execute('SELECT MAX(order_date)-MIN(order_date) FROM gold_returns').fetchone()[0]
    ok8 = span >= 30
    results.append(('8 span>=30d', ok8))
    print(f'  {"OK" if ok8 else "XX"}  8 span>=30d: {"PASS" if ok8 else "FAIL"} ({span} days)')
    passed = sum(1 for _,p in results if p)
    print(f'  {passed}/{len(results)} checks passed')
    return results

run_all_checks('clean data')
print('\n--- Injecting bad row ---')
con5.execute("INSERT INTO gold_sales VALUES (DATE '2024-01-01','North','Electronics',-999.99,5)")
run_all_checks('bad-row injected')
print('\n--- Removing bad row ---')
con5.execute('DELETE FROM gold_sales WHERE total_revenue = -999.99')
run_all_checks('after cleanup')


In [ ]:
# K 7.W.6 — Inline chart verification (Streamlit app.py is in this folder)
import pandas as pd, plotly.express as px, os
from IPython.display import HTML, display as ipy_display
BASE = os.path.abspath('..')

sales   = pd.read_parquet(os.path.join(BASE, 'gold', 'daily_sales_by_category.parquet'))
returns = pd.read_parquet(os.path.join(BASE, 'gold', 'returns_rate.parquet'))
sales['order_date']   = pd.to_datetime(sales['order_date'])
returns['order_date'] = pd.to_datetime(returns['order_date'])

# Chart 1 — revenue by region
rev_agg = sales.groupby(['region','product_category'], as_index=False)['total_revenue'].sum()
fig1 = px.bar(rev_agg, x='region', y='total_revenue', color='product_category',
              barmode='group', title='Total Revenue by Region & Category',
              labels={'total_revenue':'Revenue ($)','region':'Region'})
ipy_display(HTML(fig1.to_html(include_plotlyjs='cdn', full_html=False)))

# Chart 2 — returns rate over time
avg_rr = returns['returns_rate_pct'].mean()
fig2 = px.line(returns, x='order_date', y='returns_rate_pct',
               title='Returns Rate Over Time (%)',
               labels={'returns_rate_pct':'Returns Rate (%)','order_date':'Date'})
fig2.add_hline(y=avg_rr, line_dash='dot', annotation_text='Average', line_color='grey')
ipy_display(HTML(fig2.to_html(include_plotlyjs='cdn', full_html=False)))

# Metric cards
print(f'Total Revenue:      ${sales["total_revenue"].sum():,.0f}')
print(f'Avg Returns Rate:   {avg_rr:.1f}%')
print(f'Data last updated:  {sales["order_date"].max().date()}')
print(f'\nNote: one improvement — add a "pending orders excluded" annotation to Chart 2')
print('Run: streamlit run app.py  (from kata-workspace/)')


In [14]:
# K 7.W.7 — Agent hand-off: course-completions pipeline
# (Full pipeline in course-pipeline.py; summary below)
import subprocess, sys, os
BASE = os.path.abspath('..')
result = subprocess.run(
    [sys.executable, os.path.join(BASE, 'course-pipeline.py')],
    capture_output=True, text=True, cwd=BASE
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

print('\n--- comparison.md summary ---')
with open(os.path.join(BASE, 'comparison.md'), encoding='utf-8') as f:
    lines = f.readlines()
for line in lines[:30]:
    print(line, end='')


BRONZE  rows=500, null_completion=20, dup_event_ids=9
SILVER  rows=472 (expected 472), null_completion=0, dup_events=0
  Math: 500 - 20 - 8 = 472 => PASS
GOLD    completions rows=307 grain=PASS, dropout range=0.0..100.0%

DQ CHECKS:
  OK  1  No null event_date: PASS
  OK  2  avg_completion_pct in [0, 100]: PASS
  OK  3  completion_count > 0: PASS
  OK  4  No duplicate grain (date+category): PASS
  OK  5  dropout_rate_pct in [0, 100]: PASS
  OK  6  date span >= 30 days: PASS (364 days)

  6/6 checks passed

SERVING 5 categories, dropout rate range: 0.0%..100.0%
Course charts written to kata-workspace/

K 7.W.7 pipeline run complete


--- comparison.md summary ---
# K 7.W.7 — Agent Hand-off Comparison

**Date:** 2026-07-02  
**Agent:** Claude Code (this session)  
**Pattern handed off:** bronze→silver→gold→DQ→serve, built in K 7.W.1–7.W.6  
**New dataset:** Online course completion events (500 rows, 5 categories, mixed date formats)

---

## One time-saving

**The agent built the full br